# Week 3 prerequisites — start here

Use this notebook **before Day 1**, or when recovering from the missing `gaia_clean_day1.csv` error. It checks your Python environment, loads the current classification helper, and verifies that the Gaia sample is ready.

The Gaia sample has already been downloaded in this workspace. The cells below reuse that cache. If you remove the cache, the loader fetches a real sample again, which requires internet access.

**Finish line:** the last code cell prints `READY FOR WEEK 3`. Then open [Day 1: Classification target](week3_day_01_classification_target.ipynb).

## Restart the kernel and run the notebook in VS Code

A **kernel** is the Python process running your notebook. It remembers imported code and variables. After `classification.py` changes, restart it so Python loads the updated version. Restarting clears variables in memory; it does not delete your notebooks or cached CSV.

1. Open this `.ipynb` file in VS Code's notebook editor.
2. In the **top-right corner**, click **Select Kernel** (or the current Python/kernel name). Choose **Python Environments**, using **Select Another Kernel** first if shown, then select the environment whose Python path is `gaia-explorer/venv/bin/python`.
3. In the **toolbar above the notebook cells**, click **Restart** (the circular-arrow icon). Confirm the restart if prompted. If the button is hidden, check the toolbar's **…** overflow menu.
4. Once the kernel is ready, click **Run All** in the same notebook toolbar. This executes code cells from top to bottom. Do not run only the cell that previously failed: it needs the imports and variables from earlier cells.
5. Wait for execution to finish. A running cell shows a spinner or `[*]`; completed code cells show an execution count. The final cell below should print **READY FOR WEEK 3**.

If this file opens as raw JSON, use **Reopen Editor With… → Jupyter Notebook**. If the Python environment is unavailable, use the setup instructions below.

[VS Code notebook toolbar documentation](https://code.visualstudio.com/docs/datascience/jupyter-notebooks) · [Kernel selection documentation](https://code.visualstudio.com/docs/datascience/jupyter-kernel-management)

### If you use JupyterLab instead

With the notebook open, use **Kernel → Restart Kernel and Run All Cells…**, then confirm. This combines the restart and execution steps. [JupyterLab command reference](https://jupyterlab.readthedocs.io/en/v4.5.x/user/commands.html)


## One-time environment setup (only if needed)

Open **Terminal → New Terminal** in VS Code and run these shell commands there, not in a Python cell:

```bash
cd ~/Test/gaia-explorer
# Only if the venv directory does not exist:
python3 -m venv venv
# Install the project's dependencies in that environment:
venv/bin/python -m pip install -r requirements.txt
```

Then select `gaia-explorer/venv/bin/python` as the notebook kernel. If it does not appear, open the Command Palette (`Cmd+Shift+P` on Mac), choose **Python: Select Interpreter**, and enter that interpreter path; reopen the notebook's kernel picker.

For JupyterLab, launch it from the project directory with `venv/bin/jupyter lab`.

## 1. Check the environment and load the latest helper

Run this cell with its ▶ button or **Shift+Enter**, or use **Run All** as described above. The Python path should point inside this project's `venv`. The helper path should end in `gaia-explorer/src/classification.py`.

The reload below also refreshes the helper if this notebook's kernel already imported it. Other open notebooks can have separate kernels; restart and Run All in the Day 1 notebook too.

`ROOT` is the project folder. If Jupyter starts in `notebooks`, we move up one folder. `sys.path.insert` lets Python find `src`. The `data` module only loads/cleans measurements; the ML calculations are visible in the daily notebooks.

In [ ]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd
from IPython.display import display

# Jupyter may start in the project folder or in its notebooks folder.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src/classification.py').is_file():
    raise RuntimeError('Open Jupyter from gaia-explorer or gaia-explorer/notebooks.')
sys.path.insert(0, str(ROOT))
from src import classification as data, visualize
# Refresh the data loader if this kernel imported an older version.
data = importlib.reload(data)
visualize = importlib.reload(visualize)  # Pick up updated plot labels and layout.

DATA_PATH = None  # None uses the shared W2 cache; a missing cache is downloaded.
OUTPUT = ROOT / 'data/processed/week3'
OUTPUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)

print('Python:', sys.executable)
print('Project:', ROOT)
print('Data helper:', data.__file__)


## 2. Check the data path

Leave `DATA_PATH = None` to use the shared W2 cache. You do **not** need to run W2 first. If you already have another Gaia CSV, set its path here (for example `Path("~/Downloads/stars.csv")`). An explicit path must point to an existing file; the loader will not replace a mistyped filename with a download.

Your CSV needs `source_id`, `bp_rp`, `phot_g_mean_mag`, `parallax`, and `parallax_error`. If you use a custom path, set the same `DATA_PATH` in Day 1. Later days read the prepared data saved by Day 1.

In [ ]:
cache_path = Path(DATA_PATH).expanduser() if DATA_PATH is not None else ROOT / "data/processed/gaia_clean_day1.csv"
print("CSV:", cache_path)
print("Cached file exists:", cache_path.is_file())
print("The next cell will reuse this file." if cache_path.is_file()
      else "The next cell will fetch Gaia data if DATA_PATH is None; otherwise correct the custom path.")


## 3. Load the sample and confirm readiness

The first download can take a little time; wait for the cell to finish. Cached runs work offline. For an explicitly offline check, change `use_query_if_missing=True` to `False`.

The loader checks measurement quality. Day 1 will create labels, check class counts, and save the split. This setup check does not train a model or score the reserved test set.

In [ ]:
gaia, audit = data.load_week3_data(DATA_PATH, use_query_if_missing=True)
display(audit)
print(f'Usable Gaia measurements: {len(gaia):,}')
print('READY FOR WEEK 3 — Day 1 will create labels and save the data split.')


## About the old line 76 error

You do **not** need that `FileNotFoundError`. It came from an older version of `load_week3_data` still held in your notebook kernel. The CSV is now cached and the current loader can read it. In the daily notebook where you saw that message, run the **Setup** cell again or use **Restart → Run All**. This prerequisites notebook prints the `classification.py` path so you can confirm it uses this project. A traceback already shown on screen can remain visible until you rerun or clear that cell's output.

## If something still fails

| What you see | What to do |
|-------------|------------|
| Old `FileNotFoundError` from the earlier loader | Restart the kernel in the notebook showing the error, then **Run All**. Confirm its helper path points to this project's `src/classification.py`. |
| `No module named …` | Select the project's `venv` kernel and install `requirements.txt` using the terminal commands above. |
| `No Gaia CSV at …` with a custom path | Correct `DATA_PATH`, or set it to `None` to use the shared cache. |
| `CERTIFICATE_VERIFY_FAILED` while downloading | The current cache can be used offline. For a new download on the managed network, configure `SSL_CERT_FILE` and `REQUESTS_CA_BUNDLE` to an existing trusted CA bundle **before importing the Gaia fetch code**, then restart and retry. The loader verifies certificates by default. |
| Network/archive timeout | Check the network/VPN and retry later, or point `DATA_PATH` at an existing Gaia CSV. |
| Not enough sources in both classes | Use a larger Gaia sample containing both brightness classes. |

After `READY FOR WEEK 3`, open [Day 1](week3_day_01_classification_target.ipynb), select the same Python environment, and click **Restart**, then **Run All**. Continue through Days 2–7 in order. Day 1 saves learning_data.csv. Days 2–7 load that file, and Day 7 also loads the model choice saved by Day 6. They do not rely on variables from this prerequisites notebook.